[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Sessions


## What you will be able to do

Open a session, add a hero, and know exactly when the database sees it and when the id arrives. Read
what a commit does to the objects you are holding, and pick between the two ways of keeping one
usable afterwards. Change a hero and delete one, find out what `get` does when the row is not there,
and leave a block without committing on purpose. Recognize the four failures around a session: the
attribute read after its session closed, the session that refuses everything after one failed flush,
the row a model built with a misspelled keyword, and the team that still has heroes in it.


## The idea

### The problem

The classes describe the tables and the engine built them, and so far every row has been written by
handing a dictionary to an `INSERT`. That is not how a program works with objects. A program makes a
`Hero`, changes a hero it read a moment ago, deletes another, and wants all three to happen together
or not at all, with the ids filled in and the objects still usable afterwards.

Doing that by hand means keeping track of a great deal. Which objects are new, which have changed and
which are to be deleted; what order to write them in, so that a hero is not written before the team
its `team_id` points at; what the database gave each new row for an id; and how to undo the lot if
one statement fails halfway.

A session does that bookkeeping. You hand it objects, it holds them, and at the commit it works out
the statements, sends them in an order the foreign keys accept, and copies the new ids back. What it
asks in return is that you understand two moments: the commit, which writes, and the close, which
lets go. Most of the surprises in this notebook happen because something was read after one of them.

### What a session is

> A **session** is a workspace for objects and the transaction behind it. **`add`** puts an object in
> it, **`commit`** writes everything it holds and ends the transaction, **`get`** loads one row by
> primary key, and **`delete`** marks an object for removal. A session is used in a `with` block, and
> leaving the block **closes** it, rolling back anything not committed. A commit **expires** every
> object the session holds, so the next read of an attribute goes to the database for it, which is
> why an object read after its session closed raises `DetachedInstanceError` unless it was refreshed
> or the session was made with `expire_on_commit=False`.

### Why it works that way

- **Nothing is written until the commit.** `add` puts an object in the session and sends no SQL, and
  the id stays `None` until the database supplies it.
- **A commit expires what it holds.** Another program may have changed the same rows, so the values
  in memory are no longer to be trusted, and SQLAlchemy throws them away rather than let you read
  something stale. The next read reloads.
- **Reloading needs a session.** Once the block ends there is none, so an expired attribute cannot be
  read at all. That is `DetachedInstanceError`, and it is not about the object being broken.
- **A failed flush poisons the session.** After a statement fails mid-transaction, the session
  refuses to do anything else until `rollback()`, because carrying on inside a transaction the
  database has already given up on would write nonsense.
- **A session is short.** One per request, one per job, one per `with` block. A session kept open for
  hours holds a connection and a transaction with it.

### Where this shows up

Every write a program makes. A FastAPI route opens one session for one request and closes it at the
end, which the **SQLModel in FastAPI** notebook builds; a loader opens one per batch; a test opens
one and rolls it back. The **SQLAlchemy, Deep Dive** guide's The Session and The Identity Map
notebooks are the long version of this one, with the unit of work, the identity map and flush order
in full.

### What this notebook covers

- A session, `add`, and the id that arrives at the commit
- What the commit did to the object you are holding
- Two ways to keep an object usable after the block
- A block that never commits
- `get`, a change, and a delete
- Heroes saved safely, finished
- Four failures, from an attribute read too late to a team that still has heroes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, Session, SQLModel, create_engine


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    secret_name: str


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)

with Session(engine) as session:
    hero = Hero(name="Deadpond", secret_name="Dive Wilson")
    session.add(hero)
    print("after add   :", hero.id)
    session.commit()
    print("after commit:", hero.id)
    print("read back   :", session.get(Hero, 1).name)
```

```
after add   : None
after commit: 1
read back   : Deadpond
```

`add` did not write anything: the hero was in the session, and its id was still `None` because no
database had seen it. The commit sent the `INSERT`, and reading `hero.id` afterwards went back to the
database and came back with the 1 that SQLite gave the row.


## Setup

Twelve imports, one of them installed first where it is missing, the cast, two helpers, the printer
for what an engine logs, the classes, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Session`, `create_engine` and `select`, from
  it, are the classes, the session and the engine. Colab does not have SQLModel, so the cell installs
  0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`, from
  `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `event`, `insert`, `inspect` and `text`, from `sqlalchemy`, are what `hero_engine` and `build`
  need: the pragma on every connection, the rows loaded without a session, and what the session has
  loaded or expired
- `IntegrityError` and `PendingRollbackError`, from `sqlalchemy.exc`, and `DetachedInstanceError`,
  from `sqlalchemy.orm.exc`, are the three this notebook catches by name
- `logging` carries the SQL an engine logs to `PrintStatements`
- `re` takes memory addresses out of a message, which `DetachedInstanceError` carries, `Path` names
  the database file, and `shutil` removes the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, and `build` loads all of it

`build` creates the tables and loads the eight heroes and three teams on a connection, with
`insert`, and not through a session, so that Setup does not use what this notebook is about to teach.
`hero_engine` is the **Engine and create_all** notebook's engine, with foreign keys checked on every
connection, which the last of the Common errors depends on.


In [1]:
import logging
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, insert, inspect, text
from sqlalchemy.exc import IntegrityError, PendingRollbackError
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with engine.connect() as connection:
    print("sqlmodel", sqlmodel.__version__, "|",
          connection.execute(text("select count(*) from hero")).scalar(), "heroes in",
          connection.execute(text("select count(*) from team")).scalar(), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


## Worked examples

### A session, add, and the id that arrives at the commit

A session is opened on the engine, in a `with` block so that it is closed however the block ends:


In [2]:
with Session(engine) as session:
    ghost = Hero(name="Ghost Girl", secret_name="Ana Vega", age=27, team_id=1)
    session.add(ghost)
    print("id after add :", ghost.id)
    print("waiting to be written:", [hero.name for hero in session.new])

    engine.echo = True
    session.commit()
    engine.echo = False

    print("id after commit:", ghost.id)


id after add : None
waiting to be written: ['Ghost Girl']
    BEGIN (implicit)
    INSERT INTO hero (name, secret_name, age, team_id) VALUES (?, ?, ?, ?)
    values: ('Ghost Girl', 'Ana Vega', 27, 1)
    COMMIT
id after commit: 9


`add` sent nothing. The hero sat in `session.new`, the set of objects the session will insert, and
its id was `None` because no database had seen it. The commit did the work: `BEGIN`, one `INSERT`
with the values, `COMMIT`, and then, because reading `ghost.id` needed a value the session had just
thrown away, one `SELECT` to fetch the row back. That last query is the subject of the next section.

### What the commit did to the object you are holding

A commit expires every object the session holds. `inspect(...).unloaded` names the attributes that
would have to be fetched before they can be read:


In [3]:
with Session(engine) as session:
    ice = Hero(name="Lady Ice", secret_name="Kara Frost", age=29, team_id=1)
    session.add(ice)
    print("before the commit:", sorted(inspect(ice).unloaded))
    session.commit()
    print("after the commit :", sorted(inspect(ice).unloaded))
    print("reading one      :", ice.age)
    print("and now          :", sorted(inspect(ice).unloaded))


before the commit: []
after the commit : ['age', 'id', 'name', 'secret_name', 'team_id']
reading one      : 29
and now          : []


Before the commit every value was in memory, where the program had put it. After the commit not one
of them is: the row is written, other connections may change it at any moment, and SQLAlchemy would
rather fetch it again than hand back something stale. Reading a single attribute fetched the whole
row, which is why the last line shows nothing left unloaded.

Inside the block that is invisible, because there is always a session to fetch with. After the block
there is not: the values are gone, there is nothing to fetch them with, and reading one raises
`DetachedInstanceError`, which the first of the Common errors shows in the shape it usually takes.

### Two ways to keep an object usable after the block

The first is to read it before the block ends, which is what `refresh` does deliberately: one
`SELECT` that loads the row back into the object.


In [4]:
with Session(engine) as session:
    stone = Hero(name="Stone Hand", secret_name="Marco Pietra", age=38, team_id=2)
    session.add(stone)
    session.commit()
    session.refresh(stone)                                          # load the row back, while there is a session

print("after the block:", fields(stone))


after the block: {'id': 11, 'name': 'Stone Hand', 'secret_name': 'Marco Pietra', 'age': 38, 'team_id': 2}


The second is to tell the session not to expire anything at the commit, which it will not do again
for the life of that session:


In [5]:
with Session(engine, expire_on_commit=False) as session:
    fox = Hero(name="Silver Fox", secret_name="Vera Lux", team_id=None)
    session.add(fox)
    session.commit()
    print("unloaded after the commit:", sorted(inspect(fox).unloaded))

print("after the block:", fields(fox))


unloaded after the commit: []
after the block: {'id': 12, 'name': 'Silver Fox', 'secret_name': 'Vera Lux', 'age': None, 'team_id': None}


| Which to use | When | What it costs |
|---|---|---|
| `session.refresh(obj)` | one object is needed after the block, and it must hold what the database has | one `SELECT` per object |
| `expire_on_commit=False` | a route or a job that commits and then serializes what it wrote | the objects may be stale if something else changed those rows |
| neither | the objects are not read after the commit | nothing |

`expire_on_commit=False` is the one a service usually wants: a route commits a hero and then has to
build a response out of it, and refreshing every object by hand to do that is noise. The
**SQLModel in FastAPI** notebook uses it for exactly that.

### A block that never commits

Leaving a session's block does not write what is in it. It closes the session, and the transaction
that was open goes back:


In [6]:
with Session(engine) as session:
    session.add(Hero(name="Mystery Man", secret_name="Unknown", age=44, team_id=1))
    print("in the session:", [hero.name for hero in session.new])

with Session(engine) as session:
    print("in the database:", [hero.name for hero in session.exec(select(Hero)) if hero.name == "Mystery Man"])


in the session: ['Mystery Man']
in the database: []


Nothing was written, and nothing said so. That is the right behavior, since a block that ended on an
error must not leave half a change behind, and it is worth knowing for the times a commit is simply
forgotten. `select` and `exec` are the **Reading Rows** notebook's subject; here they only show what
is in the table.

### get, a change, and a delete

`get` takes the class and a primary key, and returns `None` rather than raising when there is no such
row:


In [7]:
with Session(engine) as session:
    print("hero 3   :", session.get(Hero, 3).name)
    print("hero 999 :", session.get(Hero, 999))

    deadpond = session.get(Hero, 1)
    deadpond.age = 31                                               # a plain attribute assignment
    print("changed  :", [hero.name for hero in session.dirty])
    session.commit()
    print("age now  :", session.get(Hero, 1).age)


hero 3   : Rusty-Man
hero 999 : None
changed  : ['Deadpond']
age now  : 31


Nothing here says `UPDATE`. The session had loaded the hero, so it knows what the row held; changing
an attribute puts the object in `session.dirty`, and the commit writes the one column that differs.
A delete is the same shape, with the object handed to `delete`:


In [8]:
with Session(engine) as session:
    mystery = session.get(Hero, 9)
    print("deleting:", mystery.name, "| id", mystery.id)
    session.delete(mystery)
    session.commit()
    print("gone    :", session.get(Hero, 9))
    print("heroes  :", len(session.exec(select(Hero)).all()))


deleting: Ghost Girl | id 9
gone    : None
heroes  : 11


Hero 9 is Ghost Girl, the first hero this notebook added, and the count is the eight the cast started
with plus the three that are still here. A delete is one row here; a delete of a
team that heroes point at is a different matter, which the last of the Common errors shows and the
**Relationships** notebook answers with `cascade_delete`.

### Heroes saved safely, finished

The pieces of this notebook in one function. `save_hero` takes raw data, validates it into a `Hero`,
writes it in a session of its own, and returns a hero that is still usable afterwards, or the reason
the database refused it:


In [9]:
def save_hero(engine, raw):
    """Validate and write one hero, returning it and its refusal, whichever happened."""
    with Session(engine, expire_on_commit=False) as session:        # so the hero survives the block
        hero = Hero.model_validate(raw)
        session.add(hero)
        try:
            session.commit()
        except IntegrityError as error:
            session.rollback()                                      # the session is usable again
            return None, str(error).splitlines()[0]
        return hero, None


for raw in [{"name": "Steel Wren", "secret_name": "Nadia Vance", "age": "39", "team_id": 1},
            {"name": "Nobody", "secret_name": "No Team", "team_id": 999}]:
    hero, refused = save_hero(engine, raw)
    print(fields(hero) if hero else f"refused: {refused}")


{'id': 13, 'name': 'Steel Wren', 'secret_name': 'Nadia Vance', 'age': 39, 'team_id': 1}
refused: (sqlite3.IntegrityError) FOREIGN KEY constraint failed


The first hero was written, and is readable after its session closed because that session does not
expire what it commits. The second names a team that does not exist, so the database refused the
`INSERT`, and the function rolled back and reported it rather than leaving a poisoned session behind
for the next caller. The age that arrived as the text `"39"` is a number, which `model_validate` did
on the way in, as the **table=True** notebook showed.

### Where each part came from

| In `save_hero` | What it relies on | The section that showed it |
|---|---|---|
| `Session(engine, expire_on_commit=False)` | a session whose commit leaves its objects readable | Two ways to keep an object usable after the block |
| `Hero.model_validate(raw)` | the Pydantic half of a table model | the **table=True** notebook |
| `session.add(hero)` | an object held, with no SQL sent yet | A session, add, and the id that arrives at the commit |
| `session.commit()` | one transaction, and the ids copied back | A session, add, and the id that arrives at the commit |
| `except IntegrityError` and `rollback()` | a refusal from the database, and a session made usable again | Common errors, below |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/03-sessions-solutions.ipynb).

**1.** Add a team, Wakaland Guard's rivals, called Grand Council with headquarters Grand Hall, and
print its id before and after the commit.


In [10]:
# your code here


**2.** In one session, add two heroes and commit once. Print how many `INSERT` statements the commit
sent, with `echo`.


In [11]:
# your code here


**3.** Load hero 2, print what `inspect(...).unloaded` says before and after a commit that changes
the hero's age, and explain the difference in a comment.


In [12]:
# your code here


**4.** Write a function that returns a hero by id from a session of its own, so that the caller can
read its name. Do it without `expire_on_commit=False`.


In [13]:
# your code here


**5.** Move every hero on the Preventers to the Z-Force in one session, and print how many rows
changed. Then check the move in a second session.


In [14]:
# your code here


**6.** Delete a team that has no heroes and commit. Then try to delete one that has heroes, print
what the database says, and leave the session usable.


In [15]:
# your code here


## Common errors

### sqlalchemy.orm.exc.DetachedInstanceError: Instance <Hero at 0x...> is not bound to a Session; attribute refresh operation cannot proceed


In [16]:
def add_hero(engine, **values):
    """Write one hero and hand it back to the caller."""
    with Session(engine) as session:
        hero = Hero(**values)
        session.add(hero)
        session.commit()                                            # every attribute of hero is expired here
        return hero


vale = add_hero(engine, name="Iron Vale", secret_name="Nina Cross", age=34, team_id=2)
try:
    print(vale.name)
except DetachedInstanceError as error:                              # its message names a memory address
    print(type(error).__name__ + ":", message(error))


DetachedInstanceError: Instance <Hero at 0x...> is not bound to a Session; attribute refresh operation cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)


This is the shape the error usually arrives in: a function, or a FastAPI route, that writes a hero
and returns it, and a caller that reads a field off what came back. The commit expired every
attribute, the block then closed the session, and the caller's first read has nothing to fetch with.

The hero was written: the commit succeeded, and only the caller's read failed, which is worth
knowing before reaching for the fix. A hero read with `get` and never committed does not do this,
because `get` loaded every column and nothing expired them afterwards. It is the commit that empties
the object, so the fix goes in the function that commits: refresh before returning, or give the
session `expire_on_commit=False`.


In [17]:
def add_hero(engine, **values):
    """Write one hero and hand it back, with its values loaded."""
    with Session(engine) as session:
        hero = Hero(**values)
        session.add(hero)
        session.commit()
        session.refresh(hero)                                       # one SELECT, while there is a session
        return hero


hawk = add_hero(engine, name="Copper Hawk", secret_name="Ada Reyes", age=26, team_id=2)
print(fields(hawk))


{'id': 15, 'name': 'Copper Hawk', 'secret_name': 'Ada Reyes', 'age': 26, 'team_id': 2}


### sqlalchemy.exc.PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush.


In [18]:
with Session(engine) as session:
    session.add(Hero(id=1, name="Copy", secret_name="Of Deadpond"))  # id 1 is taken
    try:
        session.commit()
    except IntegrityError as error:
        print("the commit:", str(error).splitlines()[0])

    session.get(Hero, 1)                                            # anything at all, on the same session


the commit: (sqlite3.IntegrityError) UNIQUE constraint failed: hero.id


PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (sqlite3.IntegrityError) UNIQUE constraint failed: hero.id
[SQL: INSERT INTO hero (id, name, secret_name, age, team_id) VALUES (?, ?, ?, ?, ?)]
[parameters: (1, 'Copy', 'Of Deadpond', None, None)]
(Background on this error at: https://sqlalche.me/e/20/gkpj) (Background on this error at: https://sqlalche.me/e/20/7s2a)

The first failure is the real one: a primary key that is already in the table. The second is what
makes it confusing in a program, because the traceback a reader sees is often this one, from a line
that has nothing wrong with it, several functions away from the mistake.

A session whose flush failed refuses everything until it is rolled back. Roll it back where the
failure is caught, and it works again:


In [19]:
with Session(engine) as session:
    session.add(Hero(id=1, name="Copy", secret_name="Of Deadpond"))
    try:
        session.commit()
    except IntegrityError:
        session.rollback()                                          # the transaction is gone; the session is not
    print("still usable:", session.get(Hero, 1).name)


still usable: Deadpond


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: hero.secret_name


In [20]:
with Session(engine) as session:
    session.add(Hero(name="Spider-Boy", secet_name="Pedro Parqueador"))   # secret_name, misspelled
    session.commit()


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: hero.secret_name
[SQL: INSERT INTO hero (name, secret_name, age, team_id) VALUES (?, ?, ?, ?)]
[parameters: ('Spider-Boy', None, None, None)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The **table=True** notebook showed a table model's constructor dropping a keyword it has no field
for, without a word. This is where that lands: the field was never set, the column says `NOT NULL`,
and the database refuses the row. The line that made the mistake is the `Hero(...)` call, and the
line that fails is the commit, which may be in another function entirely.

`model_validate` checks at the point the data arrives, which is the whole reason to use it on
anything from outside:


In [21]:
with Session(engine) as session:
    session.rollback()                                              # the flush above failed, so start again
    try:
        session.add(Hero.model_validate({"name": "Spider-Boy", "secet_name": "Pedro Parqueador"}))
    except Exception as error:
        print(type(error).__name__ + ":", message(error))


ValidationError: 1 validation error for Hero
secret_name
  Field required [type=missing, input_value={'name': 'Spider-Boy', 's...me': 'Pedro Parqueador'}, input_type=dict]


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed


In [22]:
with Session(engine) as session:
    preventers = session.get(Team, 1)
    print("deleting:", preventers.name)
    session.delete(preventers)
    try:
        session.commit()
    except IntegrityError as error:
        print(type(error).__name__ + ":", str(error).splitlines()[0])
    session.rollback()
    print("still there:", session.get(Team, 1).name)


deleting: Preventers
IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
still there: Preventers


Heroes point at that team, and the database will not leave them pointing at a row that is gone. This
is the foreign key checking that the **Engine and create_all** notebook turned on doing its job: with
it off, SQLite would have deleted the team and left every one of those heroes with a `team_id`
matching nothing.

What to do about it is a decision, not a bug: delete the heroes with the team, or set their `team_id`
to `None` first, or refuse. The **Relationships** notebook is where that decision is written into the
models, with `cascade_delete` and `ondelete`.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [23]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `add` holds an object and sends nothing; the commit writes everything the session holds, in one
  transaction, and copies the new ids back.
- A commit expires every object the session holds, so the next attribute read fetches the row again,
  and after the block there is no session to fetch with.
- `session.refresh(obj)` before the block ends, or `expire_on_commit=False` for the whole session,
  keeps an object readable afterwards.
- A session's block that ends without a commit writes nothing, and `get` returns `None` for a row
  that is not there rather than raising.
- A failed flush leaves the session refusing everything until `rollback()`, which is why a caught
  `IntegrityError` is always followed by one.


## What is next

The **Field Types and Defaults** notebook is about what a Python type becomes as a column: `datetime`
and the timezone SQLite gives back, `Enum`, `UUID`, money with `max_digits`, the difference between a
default in Python and one in the table, and the `dict` that has no matching SQLAlchemy type at all.


---

&#8592; **Previous:** [Engine and create_all](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/02-engine-and-create-all.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Field Types and Defaults](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/04-field-types-and-defaults.ipynb) &#8594;
